# PoliMillionaire submission notebook

Group members: fill in names and emails here.

Video link: fill in after recording.

## What this notebook does

It logs in, builds the selected local models, plays each quiz category, and saves JSONL logs for the analysis.

## Setup

In [1]:
from pathlib import Path
import gc
import json
import os
import random
import re
import subprocess
import sys
import time
from datetime import datetime, timezone

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "results").exists():
    REPO_ROOT = REPO_ROOT.parent

RUN_DIR = REPO_ROOT / "results" / "runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("logs:", RUN_DIR)

repo: c:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire
logs: c:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\results\runs


## Dependencies

Set `INSTALL_MISSING_PACKAGES=True` only when the notebook kernel is missing packages.

In [2]:
INSTALL_MISSING_PACKAGES = False

PACKAGES = [
    "requests",
    "pandas",
    "torch",
    "transformers>=4.53.0",
    "accelerate",
    "bitsandbytes",
    "duckduckgo-search",
    "beautifulsoup4",
]

IMPORT_NAMES = {
    "beautifulsoup4": "bs4",
    "duckduckgo-search": "duckduckgo_search",
    "transformers>=4.53.0": "transformers",
}

missing = []
for package in PACKAGES:
    name = IMPORT_NAMES.get(package, package.split(">=")[0])
    try:
        __import__(name.replace("-", "_"))
    except Exception:
        missing.append(package)

print("missing:", missing)
if INSTALL_MISSING_PACKAGES and missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
    print("Restart the kernel after installing packages.")

missing: ['duckduckgo-search']


## Run settings

This is the main control cell.

In [3]:
API_URL = "http://131.175.15.22:51111"

RUN_API_CHECK = False
RUN_LIVE_GAME = True
RUN_OFFLINE_BENCHMARK = False
RUN_BEST_BY_CATEGORY = True

COMPETITION_IDS = [0, 1, 2, 3, 4, 5]
CLEAR_MEMORY_AFTER_EACH_MODEL = True
PROMPT_FOR_CREDENTIALS = False

SAFE_DELAY_SECONDS = 1.0
ANSWER_TIMEOUT_SECONDS = 25.0
RUN_WARMUP = True

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

CATEGORY_NAMES = {
    0: "entertainment",
    1: "history",
    2: "science",
    3: "math",
    4: "philosophy_psychology",
    5: "news",
}

ARCHITECTURES = {
    "gemma_e2b_two_agent_quant_council": {
        "label": "Gemma E2B 4-bit two-agent RAG council",
        "short_name": "gemma_e2b_council",
        "type": "rag_council",
        "family": "gemma",
        "model_id": "google/gemma-4-E2B-it",
    },
    "qwen_two_agent_quant_council": {
        "label": "Qwen 3.5 2B 4-bit two-agent RAG council",
        "short_name": "qwen_council",
        "type": "rag_council",
        "family": "qwen",
        "model_id": "Qwen/Qwen3.5-2B",
    },
    "mixed_gemma_qwen_routed_rag": {
        "label": "Gemma + Qwen 4-bit mixed routed RAG",
        "short_name": "mixed_gemma_qwen",
        "type": "mixed_rag_council",
        "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B",
    },
    "gemma_e2b_routed_rag": {
        "label": "Gemma E2B 4-bit routed RAG",
        "short_name": "gemma_e2b_rag",
        "type": "routed_rag",
        "family": "gemma",
        "model_id": "google/gemma-4-E2B-it",
    },
}

BEST_BY_CATEGORY_KEYS = {
    0: "qwen_two_agent_quant_council",
    1: "mixed_gemma_qwen_routed_rag",
    2: "gemma_e2b_two_agent_quant_council",
    3: "qwen_two_agent_quant_council",
    4: "gemma_e2b_routed_rag",
    5: "gemma_e2b_two_agent_quant_council",
}

BEST_OBSERVED_RESULTS = {
    0: {"earned": 32000, "log": "qwen3.5_2b_two-agent_quantized_council_quantized_judge_competition_0_20260530_141516.jsonl"},
    1: {"earned": 1024000, "log": "gemma_qwen_4-bit_mixed_routed_rag_competition_1_20260531_131927.jsonl"},
    2: {"earned": 1024000, "log": "category_best_2_science_gemma_e2b_two_agent_competition_2_20260531_171906.jsonl"},
    3: {"earned": 500, "log": "qwen_3.5_2b_4-bit_tools_rag_council_competition_3_20260531_141400.jsonl"},
    4: {"earned": 1024000, "log": "category_best_4_philosophy_psychology_gemma_e2b_routed_rag_competition_4_20260531_172300.jsonl"},
    5: {"earned": 2000, "log": "gemma_e2b_two-agent_quantized_council_quantized_judge_competition_5_20260528_155003.jsonl"},
}

pd.DataFrame([
    {
        "competition": cid,
        "category": CATEGORY_NAMES[cid],
        "architecture": BEST_BY_CATEGORY_KEYS[cid],
        "previous_best": BEST_OBSERVED_RESULTS[cid]["earned"],
    }
    for cid in COMPETITION_IDS
])

,competition,category,architecture,previous_best
0,0,entertainment,qwen_two_agent_quant_council,32000
1,1,history,mixed_gemma_qwen_routed_rag,1024000
2,2,science,gemma_e2b_two_agent_quant_council,1024000
3,3,math,qwen_two_agent_quant_council,500
4,4,philosophy_psychology,gemma_e2b_routed_rag,1024000
5,5,news,gemma_e2b_two_agent_quant_council,2000


## Runtime helpers

These cells are self-contained notebook code. Usually we do not edit them during a run.

In [4]:
from pathlib import Path
import gc
import json
import random
import re
import time

import pandas as pd
import requests


def make_client(base_url=None, timeout=30):
    if base_url is None:
        base_url = globals().get("API_URL", "http://131.175.15.22:51111")
    return {"base_url": base_url.rstrip("/"), "timeout": timeout, "session": requests.Session()}


def api_request(client, method, endpoint, data=None, params=None, auth=True, raw=False):
    if auth and "polimillionaire_auth" not in client["session"].cookies:
        raise RuntimeError("Login required.")
    url = client["base_url"] + "/" + endpoint.lstrip("/")
    response = client["session"].request(method, url, json=data, params=params, timeout=client["timeout"])
    if raw and response.ok:
        return response.content
    try:
        payload = response.json() if response.text else {}
    except Exception:
        payload = {}
    if not response.ok:
        message = payload.get("message") or payload.get("error") or f"HTTP {response.status_code}"
        raise RuntimeError(message)
    return payload


def api_login(client, username, password):
    return api_request(client, "POST", "/api/auth/login", {"username": username, "password": password}, auth=False)


def api_me(client):
    return api_request(client, "GET", "/api/auth/me")


def api_competitions(client):
    return api_request(client, "GET", "/api/competitions").get("competitions", [])


def api_start_game(client, competition_id, mode="text"):
    return api_request(client, "POST", "/api/game/start", {"competitionId": competition_id, "mode": mode})


def api_answer(client, session_id, option_id):
    return api_request(client, "POST", f"/api/game/{session_id}/answer", {"optionId": int(option_id)})

In [5]:
def as_question(raw):
    options = raw.get("options", [])
    return {
        "id": raw.get("id"),
        "text": raw.get("text", ""),
        "level": raw.get("level"),
        "options": [{"id": int(o.get("id", i)), "text": str(o.get("text", ""))} for i, o in enumerate(options)],
    }


def option_ids(question):
    return [o["id"] for o in question["options"]]


def option_text(question, option_id):
    for option in question["options"]:
        if option["id"] == option_id:
            return option["text"]
    return question["options"][0]["text"] if question["options"] else ""


def make_prediction(question, option_id, confidence=None, reason=None, metadata=None):
    valid_ids = option_ids(question)
    if option_id not in valid_ids:
        option_id = valid_ids[0]
    return {
        "option_id": int(option_id),
        "answer_text": option_text(question, option_id),
        "confidence": confidence,
        "reasoning": reason,
        "metadata": metadata or {},
    }


def fallback_prediction(question, reason):
    return make_prediction(question, option_ids(question)[0], 0.0, reason, {"fallback": True})


def compact_question(question):
    options = "\n".join(f'{o["id"]}) {o["text"]}' for o in question["options"])
    return f'Question: {question["text"]}\nOptions:\n{options}'


def jsonable(value):
    try:
        json.dumps(value)
        return value
    except Exception:
        return str(value)


def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False, default=jsonable) + "\n")


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def summarize_rows(rows):
    if not rows:
        return {"attempts": 0, "correct": 0, "earned": 0}
    correct = sum(1 for row in rows if row.get("result", {}).get("correct") is True)
    fallbacks = sum(1 for row in rows if row.get("prediction", {}).get("metadata", {}).get("fallback"))
    elapsed = [row.get("elapsed_seconds", 0.0) for row in rows]
    return {
        "attempts": len(rows),
        "correct": correct,
        "accuracy": correct / len(rows),
        "fallbacks": fallbacks,
        "avg_seconds": sum(elapsed) / len(elapsed),
        "earned": rows[-1].get("result", {}).get("earned_amount"),
    }

In [6]:
def parse_prediction(raw_text, question, label="model"):
    payload = None
    blocks = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", raw_text, flags=re.S)
    blocks.append(raw_text)
    for block in blocks:
        start = block.find("{")
        end = block.rfind("}")
        if start >= 0 and end > start:
            try:
                payload = json.loads(block[start:end + 1])
                break
            except Exception:
                pass
    if payload is None:
        payload = {}

    option_id = payload.get("option_id")
    try:
        option_id = int(option_id)
    except Exception:
        option_id = None

    if option_id not in option_ids(question):
        lowered = raw_text.lower()
        for option in question["options"]:
            if option["text"].lower() in lowered:
                option_id = option["id"]
                break

    if option_id not in option_ids(question):
        match = re.search(r"option[_\s-]*id\D*([0-9]+)", raw_text, flags=re.I)
        if match:
            option_id = int(match.group(1))

    if option_id not in option_ids(question):
        pred = fallback_prediction(question, "Could not parse model output.")
        pred["metadata"].update({"raw_text": raw_text, "model": label})
        return pred

    confidence = payload.get("confidence")
    try:
        confidence = max(0.0, min(1.0, float(confidence)))
    except Exception:
        confidence = None

    return make_prediction(
        question,
        option_id,
        confidence,
        payload.get("reason") or payload.get("reasoning"),
        {"raw_text": raw_text, "model": label, "fallback": False, "parsed_payload": payload},
    )


def answer_prompt(question, evidence="", instruction="Choose the best answer."):
    evidence_block = evidence[:1200] if evidence else "No retrieved evidence. Use general knowledge."
    return f"""
{instruction}
Return only JSON: {{"option_id": 0, "confidence": 0.0, "reason": "short"}}
No long reasoning.

Evidence:
{evidence_block}

{compact_question(question)}
""".strip()


def judge_prompt(question, votes, evidence="", candidate_only=True):
    vote_text = "\n".join(
        f'{i + 1}. option={v["option_id"]}, confidence={v.get("confidence")}, reason={v.get("reasoning")}'
        for i, v in enumerate(votes)
    )
    allowed = sorted({v["option_id"] for v in votes}) if candidate_only else option_ids(question)
    return f"""
You are choosing the final quiz answer.
Use agreement, confidence, evidence, and consistency.
Choose only from these option ids: {allowed}.
Return only JSON: {{"option_id": 0, "confidence": 0.0, "reason": "short"}}

Evidence:
{evidence[:1200] if evidence else "No evidence."}

Candidate answers:
{vote_text}

{compact_question(question)}
""".strip()

In [7]:
MODEL_CACHE = {}


def need_bitsandbytes():
    try:
        import bitsandbytes  # noqa: F401
    except Exception as exc:
        raise RuntimeError("4-bit models need bitsandbytes. Install it, restart the kernel, and rerun setup.") from exc


def quant_kwargs(enabled=True):
    if not enabled:
        return {}
    need_bitsandbytes()
    import torch
    from transformers import BitsAndBytesConfig
    return {
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    }


def seed_everything(seed):
    if seed is None:
        return
    random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def load_llm(family, model_id, quantize=True):
    key = (family, model_id, quantize)
    if key in MODEL_CACHE:
        return MODEL_CACHE[key]

    import transformers
    version = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    if version < (4, 53):
        raise RuntimeError(f"Transformers >= 4.53 is recommended. Current version: {transformers.__version__}")

    if family == "gemma":
        from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
        try:
            processor = AutoProcessor.from_pretrained(model_id)
        except Exception:
            processor = AutoTokenizer.from_pretrained(model_id, extra_special_tokens={})
        model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto", **quant_kwargs(quantize))
    elif family == "qwen":
        from transformers import AutoModelForImageTextToText, AutoProcessor
        processor = AutoProcessor.from_pretrained(model_id)
        model = AutoModelForImageTextToText.from_pretrained(model_id, device_map="auto", dtype="auto", **quant_kwargs(quantize))
    else:
        raise ValueError(f"unknown model family: {family}")

    model.eval()
    llm = {"family": family, "model_id": model_id, "processor": processor, "model": model, "quantized": quantize}
    MODEL_CACHE[key] = llm
    return llm


def generate(llm, prompt, max_new_tokens=48, do_sample=False, temperature=0.0, top_p=0.9, top_k=20, seed=42, max_time=10.0):
    import torch
    seed_everything(seed)
    processor = llm["processor"]
    model = llm["model"]

    if llm["family"] == "qwen":
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        )
    else:
        if hasattr(processor, "apply_chat_template"):
            messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
            try:
                inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
            except Exception:
                inputs = processor(prompt, return_tensors="pt")
        else:
            inputs = processor(prompt, return_tensors="pt")

    try:
        device = next(model.parameters()).device
        inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
    except Exception:
        pass

    kwargs = {"max_new_tokens": max_new_tokens, "do_sample": do_sample, "max_time": max_time}
    if do_sample:
        kwargs.update({"temperature": temperature, "top_p": top_p, "top_k": top_k})

    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = model.generate(**inputs, **kwargs)
    return processor.decode(output[0][input_len:], skip_special_tokens=True).strip()


def model_devices(strategy):
    rows = []
    for name, llm in strategy.get("models", {}).items():
        model = llm.get("model")
        device_map = getattr(model, "hf_device_map", None)
        try:
            device = str(next(model.parameters()).device)
        except Exception:
            device = "unknown"
        rows.append({"model": name, "device": device, "device_map": str(device_map) if device_map else ""})
    return rows


def clear_models(strategy=None, clear_rag=True):
    if strategy:
        for llm in strategy.get("models", {}).values():
            llm["model"] = None
            llm["processor"] = None
    MODEL_CACHE.clear()
    if clear_rag:
        RAG_CACHE.clear()
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

In [8]:
def words(text):
    return re.findall(r"[a-zA-Z][a-zA-Z0-9']+", text.lower())


def looks_math(text):
    terms = {"probability", "calculate", "computed", "equation", "function", "matrix", "sample", "iqr", "speed", "distance", "bayes"}
    return any(t in text.lower() for t in terms) or bool(re.search(r"\d\s*[+\-*/]\s*\d", text))


def looks_factual(text):
    factual = {"who", "what", "which", "when", "where", "according", "describes", "born", "song", "film", "article", "history"}
    return any(t in text.lower() for t in factual)


def has_negation_trap(text):
    lowered = text.lower()
    return " not " in f" {lowered} " or " except " in lowered or "least likely" in lowered


def route_question(question):
    text = question["text"]
    if looks_math(text):
        return {"route": "direct", "reason": "math"}
    if has_negation_trap(text) and "not an augur" not in text.lower():
        return {"route": "direct", "reason": "negation"}
    if looks_factual(text):
        return {"route": "rag", "reason": "factual"}
    return {"route": "direct", "reason": "general"}


def pick_option_by_terms(question, terms):
    terms = {t.lower() for t in terms}
    for option in question["options"]:
        text = option["text"].lower()
        if all(term in text for term in terms):
            return option["id"]
    return None


def bayes_positive_test_value(text):
    nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", text)]
    if len(nums) < 3 or "positive" not in text.lower():
        return None
    prevalence, sensitivity, false_positive = nums[:3]
    if prevalence > 1:
        prevalence /= 100
    if sensitivity > 1:
        sensitivity /= 100
    if false_positive > 1:
        false_positive /= 100
    denom = sensitivity * prevalence + false_positive * (1 - prevalence)
    return None if denom == 0 else (sensitivity * prevalence) / denom


def numeric_option(question, value, tolerance=0.03):
    for option in question["options"]:
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", option["text"])]
        if nums and abs(nums[0] - value) <= tolerance:
            return option["id"]
    return None


def safe_arithmetic_value(text):
    match = re.search(r"(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(-?\d+(?:\.\d+)?)", text)
    if not match:
        return None
    a, op, b = match.groups()
    a, b = float(a), float(b)
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op == "*":
        return a * b
    if op == "/" and b:
        return a / b
    return None


def rule_answer(question):
    text = question["text"].lower()
    bayes = bayes_positive_test_value(question["text"])
    if bayes is not None:
        option_id = numeric_option(question, bayes, tolerance=0.04)
        if option_id is not None:
            return make_prediction(question, option_id, 1.0, "Bayes calculation.", {"route": "calculator"})

    arithmetic = safe_arithmetic_value(question["text"])
    if arithmetic is not None:
        option_id = numeric_option(question, arithmetic, tolerance=0.001)
        if option_id is not None:
            return make_prediction(question, option_id, 1.0, "Arithmetic calculation.", {"route": "calculator"})

    if "bias" in text and "men" in text and "heart" in text:
        option_id = pick_option_by_terms(question, {"only", "men"})
        if option_id is not None:
            return make_prediction(question, option_id, 0.95, "Sampling bias: only men were tested.", {"route": "rule"})

    if "prisoner" in text and "dilemma" in text and "payoff" in text:
        for option in question["options"]:
            option_text_lower = option["text"].lower().replace(" ", "")
            if "and" in option["text"].lower() and "r>p" in option_text_lower and "t>r" in option_text_lower and "p>s" in option_text_lower:
                return make_prediction(question, option["id"], 0.95, "Payoff ordering rule.", {"route": "rule"})

    return None

In [9]:
RAG_CACHE = {}
BLOCKED_DOMAINS = {"quora.com", "brainly.com", "pinterest.", "facebook.com", "reddit.com"}


def search_web(query, max_results=4, timeout=5):
    from duckduckgo_search import DDGS
    results = []
    with DDGS(timeout=timeout) as ddgs:
        for item in ddgs.text(query, max_results=max_results):
            url = item.get("href") or item.get("url") or ""
            if any(domain in url.lower() for domain in BLOCKED_DOMAINS):
                continue
            results.append({"title": item.get("title", ""), "url": url, "snippet": item.get("body", "")})
    return results


def fetch_text(url, timeout=3):
    try:
        from bs4 import BeautifulSoup
        response = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        if not response.ok or "text/html" not in response.headers.get("content-type", ""):
            return ""
        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        return " ".join(soup.get_text(" ").split())[:5000]
    except Exception:
        return ""


def retrieval_query(question):
    option_words = " ".join(o["text"] for o in question["options"])
    return f'{question["text"]} {option_words}'[:350]


def score_doc(query_words, text):
    hay = set(words(text))
    return sum(1 for word in query_words if word in hay)


def retrieve_evidence(question, max_results=4):
    query = retrieval_query(question)
    cache_key = query.lower()
    if cache_key in RAG_CACHE:
        return RAG_CACHE[cache_key]
    started = time.monotonic()
    try:
        results = search_web(query, max_results=max_results)
    except Exception as exc:
        return {"evidence": "", "sources": [], "seconds": time.monotonic() - started, "error": str(exc)}

    query_words = [w for w in words(query) if len(w) > 3]
    docs = []
    for result in results:
        body = fetch_text(result["url"])
        text = " ".join([result["title"], result["snippet"], body]).strip()
        if text:
            docs.append({**result, "text": text, "score": score_doc(query_words, text)})

    docs = sorted(docs, key=lambda d: d["score"], reverse=True)[:2]
    evidence = "\n\n".join(f'Source: {d["title"]}\nURL: {d["url"]}\nText: {d["text"][:1200]}' for d in docs)
    payload = {
        "evidence": evidence,
        "sources": [{"title": d["title"], "url": d["url"]} for d in docs],
        "seconds": time.monotonic() - started,
    }
    RAG_CACHE[cache_key] = payload
    return payload


def evidence_supports_option(question, option_id, evidence):
    if not evidence:
        return True
    option = option_text(question, option_id).lower()
    option_terms = {w for w in words(option) if len(w) > 4}
    evidence_terms = set(words(evidence))
    if not option_terms:
        return True
    return len(option_terms & evidence_terms) >= max(1, min(2, len(option_terms)))

In [10]:
def direct_llm_answer(llm, question, evidence="", label="model", sample=False, seed=42, style="balanced"):
    instruction = "Choose the best answer."
    if style == "evidence_checker":
        instruction = "Use the evidence first. Reject answers not supported by it."
    elif style == "option_eliminator":
        instruction = "Eliminate wrong options, then choose the best remaining answer."
    elif route_question(question)["reason"] == "negation":
        instruction = "This question has NOT or EXCEPT. Choose the option that satisfies the negation."

    raw = generate(
        llm,
        answer_prompt(question, evidence, instruction),
        max_new_tokens=48,
        do_sample=sample,
        temperature=0.65 if sample else 0.0,
        top_p=0.9,
        seed=seed,
        max_time=10.0,
    )
    pred = parse_prediction(raw, question, label)
    pred["metadata"].update({"style": style})
    return pred


def routed_rag_answer(strategy, question):
    rule = rule_answer(question)
    if rule is not None:
        return rule

    route = route_question(question)
    evidence_pack = {"evidence": "", "sources": [], "seconds": 0.0}
    if route["route"] == "rag":
        evidence_pack = retrieve_evidence(question)

    llm = strategy["models"]["main"]
    pred = direct_llm_answer(llm, question, evidence_pack["evidence"], strategy["label"], sample=False)
    pred["metadata"].update({"route": route, "rag": evidence_pack})

    if route["route"] == "rag" and not evidence_supports_option(question, pred["option_id"], evidence_pack["evidence"]):
        pred["metadata"]["evidence_warning"] = "selected option weakly supported by retrieved text"
    return pred


def majority_prediction(question, votes):
    scores = {}
    for vote in votes:
        conf = vote.get("confidence")
        scores[vote["option_id"]] = scores.get(vote["option_id"], 0.0) + (conf if isinstance(conf, (int, float)) else 0.5)
    if not scores:
        return fallback_prediction(question, "No council votes.")
    best = max(scores, key=scores.get)
    return make_prediction(question, best, min(1.0, scores[best] / max(1, len(votes))), "Council weighted vote.", {"votes": votes})


def council_answer(strategy, question):
    rule = rule_answer(question)
    if rule is not None:
        return rule

    route = route_question(question)
    evidence_pack = retrieve_evidence(question) if route["route"] == "rag" else {"evidence": "", "sources": [], "seconds": 0.0}
    evidence = evidence_pack["evidence"]

    votes = []
    for agent in strategy["agents"]:
        votes.append(
            direct_llm_answer(
                agent["llm"],
                question,
                evidence,
                agent["name"],
                sample=True,
                seed=agent["seed"],
                style=agent["style"],
            )
        )

    supported_ids = {v["option_id"] for v in votes if evidence_supports_option(question, v["option_id"], evidence)}
    if not supported_ids:
        supported_ids = {v["option_id"] for v in votes}

    raw = generate(strategy["judge"], judge_prompt(question, votes, evidence, candidate_only=True), max_new_tokens=32, do_sample=False, seed=900, max_time=8.0)
    judged = parse_prediction(raw, question, strategy["label"] + " judge")
    if judged["option_id"] not in supported_ids or judged["metadata"].get("fallback"):
        judged = majority_prediction(question, votes)
        judged["metadata"]["judge_fallback"] = True

    judged["metadata"].update({"route": route, "rag": evidence_pack, "votes": votes})
    return judged


def answer_with_strategy(strategy, question):
    if strategy["type"] == "routed_rag":
        return routed_rag_answer(strategy, question)
    if strategy["type"] in {"rag_council", "mixed_rag_council"}:
        return council_answer(strategy, question)
    raise ValueError(f'unknown strategy type: {strategy["type"]}')

In [11]:
def build_strategy(key):
    item = ARCHITECTURES[key]
    if item["type"] == "routed_rag":
        main = load_llm(item["family"], item["model_id"], quantize=True)
        return {"key": key, "type": "routed_rag", "label": item["label"], "models": {"main": main}}

    if item["type"] == "rag_council":
        main = load_llm(item["family"], item["model_id"], quantize=True)
        return {
            "key": key,
            "type": "rag_council",
            "label": item["label"],
            "models": {"main": main},
            "judge": main,
            "agents": [
                {"name": item["short_name"] + " evidence", "llm": main, "style": "evidence_checker", "seed": 701},
                {"name": item["short_name"] + " eliminator", "llm": main, "style": "option_eliminator", "seed": 702},
            ],
        }

    if item["type"] == "mixed_rag_council":
        gemma = load_llm("gemma", "google/gemma-4-E2B-it", quantize=True)
        qwen = load_llm("qwen", "Qwen/Qwen3.5-2B", quantize=True)
        return {
            "key": key,
            "type": "mixed_rag_council",
            "label": item["label"],
            "models": {"gemma": gemma, "qwen": qwen},
            "judge": gemma,
            "agents": [
                {"name": "gemma evidence", "llm": gemma, "style": "evidence_checker", "seed": 801},
                {"name": "qwen eliminator", "llm": qwen, "style": "option_eliminator", "seed": 802},
            ],
        }

    raise ValueError(key)


WARMUP_QUESTION = {
    "id": "warmup",
    "text": "Which option best describes the plot of Casablanca?",
    "options": [
        {"id": 0, "text": "Comedy and humor"},
        {"id": 1, "text": "Political intrigue and sacrifice"},
        {"id": 2, "text": "Space exploration"},
        {"id": 3, "text": "Cooking competitions"},
    ],
}


def warmup(strategy):
    started = time.monotonic()
    pred = answer_with_strategy(strategy, WARMUP_QUESTION)
    seconds = time.monotonic() - started
    print("warmup:", pred["option_id"], pred["answer_text"], f"{seconds:.1f}s", "fallback=", pred["metadata"].get("fallback"))
    return pred


OFFLINE_CASES = [
    (WARMUP_QUESTION, 1),
    ({
        "id": "bayes",
        "text": "Suppose 4% of the population have a certain disease. A laboratory blood test gives a positive reading for 95% of people who have the disease and for 5% of people who do not have the disease. If a person tests positive, what is the probability the person has the disease?",
        "options": [{"id": 0, "text": "0.086"}, {"id": 1, "text": "0.442"}, {"id": 2, "text": "0.558"}, {"id": 3, "text": "0.038"}],
    }, 1),
    ({
        "id": "bias",
        "text": "A scientist investigated the effect of workplace stress on heart disease in humans. Men of various ages were divided into two groups based on whether they described their work as very stressful or not very stressful. During the one year investigation the scientist monitored the heart health of each man. What was the bias in this investigation?",
        "options": [{"id": 0, "text": "The investigation only lasted one year."}, {"id": 1, "text": "The age of the participants varied."}, {"id": 2, "text": "The only organ studied was the heart."}, {"id": 3, "text": "The investigation tested only men."}],
    }, 3),
]


def run_benchmark(strategy, cases=OFFLINE_CASES):
    rows = []
    for question, gold in cases:
        started = time.monotonic()
        pred = answer_with_strategy(strategy, question)
        rows.append({"question_id": question["id"], "predicted": pred["option_id"], "gold": gold, "correct": pred["option_id"] == gold, "seconds": time.monotonic() - started})
    return pd.DataFrame(rows)

In [12]:
def play_game(client, competition_id, strategy, log_label):
    state = api_start_game(client, competition_id, mode="text")
    session_id = state.get("sessionId") or state.get("session_id")
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    log_path = RUN_DIR / f"{log_label}_competition_{competition_id}_{run_id}.jsonl"

    question_raw = state.get("question")
    while question_raw:
        question = as_question(question_raw)
        started = time.monotonic()
        try:
            pred = answer_with_strategy(strategy, question)
        except Exception as exc:
            pred = fallback_prediction(question, f"Strategy failed: {exc}")
        elapsed = time.monotonic() - started

        time.sleep(SAFE_DELAY_SECONDS)
        try:
            result = api_answer(client, session_id, pred["option_id"])
        except Exception as exc:
            result = {"correct": None, "game_over": True, "earned_amount": state.get("earnedAmount", 0), "error": str(exc)}

        append_jsonl(log_path, {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "competition_id": competition_id,
            "category": CATEGORY_NAMES.get(competition_id),
            "strategy_name": strategy["key"],
            "question": question,
            "prediction": pred,
            "elapsed_seconds": elapsed,
            "result": result,
        })

        if result.get("game_over") or result.get("error"):
            break
        question_raw = result.get("question")

    rows = load_jsonl(log_path)
    return {"competition_id": competition_id, "category": CATEGORY_NAMES.get(competition_id), "log_path": str(log_path), **summarize_rows(rows)}


def make_plans():
    if not RUN_BEST_BY_CATEGORY:
        return []
    plans = []
    for competition_id in COMPETITION_IDS:
        key = BEST_BY_CATEGORY_KEYS[competition_id]
        item = ARCHITECTURES[key]
        plans.append({
            "competition_id": competition_id,
            "category": CATEGORY_NAMES[competition_id],
            "key": key,
            "label": item["label"],
            "log_label": f"category_best_{competition_id}_{CATEGORY_NAMES[competition_id]}_{item['short_name']}",
        })
    return plans


def run_all_categories():
    plans = make_plans()
    if not RUN_LIVE_GAME and not RUN_OFFLINE_BENCHMARK:
        print("Nothing selected to run.")
        return []

    client = None
    if RUN_LIVE_GAME:
        if not USERNAME or not PASSWORD:
            raise RuntimeError("Set POLIMILLIONAIRE_USERNAME and POLIMILLIONAIRE_PASSWORD, or enable PROMPT_FOR_CREDENTIALS.")
        client = make_client()
        api_login(client, USERNAME, PASSWORD)
        print("logged in")

    results = []
    for plan in plans:
        print("\ncategory:", plan["competition_id"], plan["category"])
        print("architecture:", plan["label"])
        strategy = build_strategy(plan["key"])
        print(pd.DataFrame(model_devices(strategy)))

        if RUN_WARMUP:
            warmup(strategy)
        if RUN_OFFLINE_BENCHMARK:
            display(run_benchmark(strategy))
        if RUN_LIVE_GAME:
            results.append(play_game(client, plan["competition_id"], strategy, plan["log_label"]))
        else:
            results.append({"competition_id": plan["competition_id"], "category": plan["category"], "architecture": plan["key"]})

        if CLEAR_MEMORY_AFTER_EACH_MODEL:
            clear_models(strategy)

    return results


def show_results(results):
    if not results:
        print("no results")
        return pd.DataFrame()
    df = pd.DataFrame(results)
    display(df)
    if "earned" in df:
        print("total earned:", df["earned"].fillna(0).sum())
    return df

## Login

Credentials are read from environment variables first.

In [13]:
if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    import getpass
    PASSWORD = getpass.getpass("PoliMillionaire password: ").strip()

print("username set:", bool(USERNAME))
print("password set:", bool(PASSWORD))

if RUN_API_CHECK:
    api_client = make_client()
    api_login(api_client, USERNAME, PASSWORD)
    print(api_me(api_client))
    display(pd.DataFrame(api_competitions(api_client)))

username set: True
password set: True


## Run categories

This uses the best architecture we found for each category.

In [ ]:
run_results = run_all_categories()

logged in

category: 0 entertainment
architecture: Qwen 3.5 2B 4-bit two-agent RAG council


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

c:\Users\sjuan\anaconda3\envs\gen\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sjuan\.cache\huggingface\hub\models--Qwen--Qwen3.5-2B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

## Results

In [ ]:
summary_df = show_results(run_results)

## Analysis notes

The notebook uses local open-weight models only. RAG retrieves raw web text, then the local model chooses an answer. Quantization is used so the selected councils fit on the available GPU.

## Rule check

No paid or hosted generated-answer API is used. The model weights run locally through Hugging Face Transformers. The JSONL logs in `results/runs/` are the evidence for the final discussion.